#### RAG Evaluation

Let's evaluate the RAG which we created just now to understand how it is performing

In [ ]:
from openai import AsyncOpenAI
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness, FactualCorrectness, AnswerRelevancy, ContextRecall

creating mock ollama client, why ollama client? basically to use local ollama we need this setup to bypass it

In [ ]:
ollama_client = AsyncOpenAI(
    api_key="ollama",          # A dummy key is required, but ignored by Ollama
    base_url="http://localhost:11434/v1"
)

let's define evaluator llm

In [ ]:
evaluator_llm = llm_factory("llama3.2:1b", provider="openai", client=ollama_client)

let's load the data which we ran with our RAG

In [ ]:
sample_data = [
    {
        "user_input":"healthy snacks which is low in fat",
        "retrieved_contexts":['Name: Healthy Bites - Mini Health Bars\nCategory: Gourmet & World Food\nSub Category: Snacks, Dry Fruits, Nuts\nProduct Type: Healthy, Baked Snacks\nProduct Description: Mini Health Bars are one of the most nutritious snacks. It has got properties of different grains and nuts. These bars are a superfood for your body and brains, made with good quality ingredients. A balance of good taste and healthy munching is on your way to being added to your favourites.\xa0\n‘Snack Amor’ has launched healthy options to keep your fitness in check. The brand has come up with premium quality snacks, nuts and dried fruits collection e.t.c. Loaded with nutrients and enriching qualities these super snacks are a must-have.',
                            'Name: Snacks - Roasted Peanuts Cheese n Onion\nCategory: Gourmet & World Food\nSub Category: Snacks, Dry Fruits, Nuts\nProduct Type: Healthy, Baked Snacks\nProduct Description: Adding cheese to food makes you feel guilty? The goodness of\xa0peanuts\xa0will take the guilt away forever. The combination of fiber, fat and\xa0protein\xa0content in\xa0peanuts\xa0makes them a high satiety food. These are good sources of energy and help in increasing the metabolic rate and contribute to weight loss. Now, you need not compromise on taste when you want healthy food. Find the perfect balanced diet in Bb\xa0GoodDiet.  Frying food increases your fat and calorie content. Enjoy these completely roasted snacks and say hello to a happier and healthier you.',
                            'Name: Snacks - Roasted Diet Chivda Mixture\nCategory: Gourmet & World Food\nSub Category: Snacks, Dry Fruits, Nuts\nProduct Type: Healthy, Baked Snacks\nProduct Description: Bb GoodDiet\xa0Diet Chivda Mixture is\xa0made mainly of rice flakes, peanuts, and Bengal gram. It is a healthy meal in itself. Rice flakes popularly known as Poha is lactose-free, heart healthy and fat-free. Added to it, the peanuts and the Bengal gram make it a protein-rich diet. Its also a rich source of iron, vitamin B, carbohydrates, and proteins. What more can you ask for in a good diet?  Frying food increases your fat and calorie content. Enjoy these completely roasted snacks and say hello to a happier and healthier you.'],
        "response":'[{"name": "Healthy Bites", "category": "Gourmet & World Food", "sub_category": "Snacks, Dry Fruits, Nuts", "description": "Mini Health Bars are one of the most nutritious snacks."}, {"name": "Snacks", "category": "Gourmet & World Food", "sub_category": "Snacks, Dry Fruits, Nuts", "description": "Mini Health Bars are one of the most nutritious snacks."}]',
        "reference":'[{"name": "Healthy Bites", "category": "Gourmet & World Food", "sub_category": "Snacks, Dry Fruits, Nuts"}]'
    }
]

In [ ]:
dataset = EvaluationDataset.from_list(sample_data)

In [ ]:
faithfulness = Faithfulness(llm=evaluator_llm)

In [ ]:
result =  await faithfulness.ascore(user_input = sample_data[0]["user_input"],
        retrieved_contexts = sample_data[0]["retrieved_contexts"],
        response = sample_data[0]["response"] 
        )

In [ ]:
# WORKING
result